# 文档切分器 Text Splitters
## 为什么分割/切分/分块？
获取 Document 对象后，需要将其切分成一个个小块（Chunk）。原因：
- 长文档问题：大模型存在最大输入的 `Token 限制` ，如果一个 `Document 非常大` ，在输入大模型时会 `被截断` ，导致信息缺失。
- 检索精度：Document 可能包含 `非常多无关的信息` ，这些无效信息会 `干扰大模型` 的生成，而小块检索更精准。
- 成本控制：减少不必要的`token消耗`

无论是在存储还是检索过程中，都以这些 `块(chunk)` 为基本单位，这样能有效地避免内容噪声干扰和超出最大 Token 的问题。

## Chunking拆分的策略
方法1：根据句子切分
方法2：按照固定字符数来切分
方法3：按固定字符数来切分，结合重叠窗口
方法4：递归字符切分方法
方法5：根据语义内容切分

In [ ]:
## TextSplitter 源码分析

class TextSplitter(BaseDocumentTransformer, ABC):
    """用于将文本切分为多个块的接口。"""

    def __init__(
        self,
        chunk_size: int = 4000,
        chunk_overlap: int = 200,
        length_function: Callable[[str], int] = len,
        keep_separator: bool | Literal["start", "end"] = False,  # noqa: FBT001,FBT002
        add_start_index: bool = False,  # noqa: FBT001,FBT002
        strip_whitespace: bool = True,  # noqa: FBT001,FBT002
    ) -> None:
        """创建一个新的 `TextSplitter`。

        Args:
            chunk_size: 返回的文本块的最大大小。
            chunk_overlap: 文本块之间重叠的字符数。
            length_function: 用于衡量给定文本块长度的函数。
            keep_separator: 是否保留分隔符，以及将其放在对应文本块中的哪个位置
                `(True='start')`。
            add_start_index: 如果为 `True`，则在元数据中包含文本块的起始索引。
            strip_whitespace: 如果为 `True`，则去除每个文档开头和结尾的空白字符。

        Raises:
            ValueError: 如果 `chunk_size` 小于或等于 0。
            ValueError: 如果 `chunk_overlap` 小于 0。
            ValueError: 如果 `chunk_overlap` 大于 `chunk_size`。
        """
        if chunk_size <= 0:
            msg = f"chunk_size must be > 0, got {chunk_size}"
            raise ValueError(msg)
        if chunk_overlap < 0:
            msg = f"chunk_overlap must be >= 0, got {chunk_overlap}"
            raise ValueError(msg)
        if chunk_overlap > chunk_size:
            msg = (
                f"Got a larger chunk overlap ({chunk_overlap}) than chunk size "
                f"({chunk_size}), should be smaller."
            )
            raise ValueError(msg)

        self._chunk_size = chunk_size
        self._chunk_overlap = chunk_overlap
        self._length_function = length_function
        self._keep_separator = keep_separator
        self._add_start_index = add_start_index
        self._strip_whitespace = strip_whitespace

    @abstractmethod
    def split_text(self, text: str) -> list[str]:
        # 此方法是抽象方法，具体的实现细节由子类来决定
        """将文本切分为多个组成部分。

        Args:
            text: 要切分的文本。

        Returns:
            文本块列表。
        """

    # 传入字符串列表，返回document对象列表
    # 此方法的底层调用了split_text方法，即将参数中的每一个字符串都穿入split_text中执行，得到的字符串列表中，将字符串再封装为document对象，就构成了document对象列表
    def create_documents(
        self, 
        texts: list[str],
        metadatas: list[dict[Any, Any]] | None = None,
    ) -> list[Document]:
        """根据文本列表创建一组 `Document` 对象。

        Args:
            texts: 需要被切分并转换为文档的文本列表。
            metadatas: 可选的元数据列表，用于关联到每个文档。

        Returns:
            `Document` 对象列表。
        """
        metadatas_ = metadatas or [{}] * len(texts)
        documents = []
        for i, text in enumerate(texts):
            index = 0
            previous_chunk_len = 0
            for chunk in self.split_text(text):
                metadata = copy.deepcopy(metadatas_[i])
                if self._add_start_index:
                    offset = index + previous_chunk_len - self._chunk_overlap
                    index = text.find(chunk, max(0, offset))
                    metadata["start_index"] = index
                    previous_chunk_len = len(chunk)
                new_doc = Document(page_content=chunk, metadata=metadata)
                documents.append(new_doc)
        return documents

    # 传入的参数类型:Document对象列表，返回值类型：Document对象列表
    # 这个方法底层调用了create_documents()，将参数中的每一个document对象，提取其page_content字段，构成了字符串列表，然后调用方法2即可。
    def split_documents(self, documents: Iterable[Document]) -> list[Document]:
        """切分文档。

        Args:
            documents: 要切分的文档。

        Returns:
            切分后的文档列表。
        """
        texts, metadatas = [], []
        for doc in documents:
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)
        return self.create_documents(texts, metadatas=metadatas)

    def _join_docs(self, docs: list[str], separator: str) -> str | None:
        text = separator.join(docs)
        if self._strip_whitespace:
            text = text.strip()
        return text or None

    def _merge_splits(self, splits: Iterable[str], separator: str) -> list[str]:
        # 现在我们希望将这些较小的片段组合成中等大小的
        # 文本块，以便发送给 LLM。
        # 实现细节省略...

    @classmethod
    def from_huggingface_tokenizer(
        cls, tokenizer: PreTrainedTokenizerBase, **kwargs: Any
    ) -> TextSplitter:
        """使用 Hugging Face tokenizer 计算长度的文本切分器。

        Args:
            tokenizer: 要使用的 Hugging Face tokenizer。

        Returns:
            一个使用 Hugging Face tokenizer 进行长度计算的 `TextSplitter` 实例。
        """
        # 实现细节省略...

    @classmethod
    def from_tiktoken_encoder(
        cls,
        encoding_name: str = "gpt2",
        model_name: str | None = None,
        allowed_special: Literal["all"] | AbstractSet[str] = set(),
        disallowed_special: Literal["all"] | Collection[str] = "all",
        **kwargs: Any,
    ) -> Self:
        """使用 `tiktoken` 编码器计算长度的文本切分器。

        Args:
            encoding_name: 要使用的 tiktoken 编码名称。
            model_name: 要使用的模型名称。
                如果提供该参数，它将覆盖 `encoding_name`。
            allowed_special: 编码过程中允许的特殊 token。
            disallowed_special: 编码过程中不允许的特殊 token。

        Returns:
            一个使用 tiktoken 进行长度计算的 `TextSplitter` 实例。

        Raises:
            ImportError: 如果未安装 tiktoken 包。
        """
        # 实现细节省略...

    @override
    def transform_documents(
        self, documents: Sequence[Document], **kwargs: Any
    ) -> Sequence[Document]:
        """通过切分文档来转换文档序列。

        Args:
            documents: 要切分的文档序列。

        Returns:
            切分后的文档列表。
        """
        return self.split_documents(list(documents))

## 具体实现
### CharacterTextSplitter:Split by character
参数情况说明：
- chunk_size ：每个切块的最大字符数量，默认值为4000。
  
- chunk_overlap ：相邻两个切块之间的最大重叠字符数量，默认值为200。为了保证段之间语义完整，可以设置每个块之间有一部分重叠。
  
- separator ：分割使用的分隔符，默认值为"\n\n"。
  
- length_function ：用于计算切块长度的方法。默认赋值为父类TextSplitter的len函数。

In [ ]:
### 字符串文本的分割
### 若必须禁用分隔符（如处理无空格文本），需容忍实际块长略小于 chunk_size （尤其对中文）
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""
# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    #length_function=len,
    separator="" # 设置为空字符串时，表示禁用分隔符优先
)
# 4.分割文本
texts = splitter.split_text(text)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：49
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发
--------------------------------------------------
块 2:长度：23
象，使开发者
能够更容易地构建复杂的应用程序。
--------------------------------------------------


In [9]:
# 指定分割符
# 1.导入相关依赖
# 指定分割符
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"
# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=30,  # 每个块的最大字符数
    chunk_overlap=5,  # 块之间的重叠字符数
    separator="。",  # 按句号分割优先
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for i, chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

Created a chunk of size 33, which is longer than the specified 30


块 1:长度：9
这是一个示例文本啊
--------------------------------------------------
块 2:长度：33
我们将使用CharacterTextSplitter将其分割成小块
--------------------------------------------------
块 3:长度：7
分割基于字符数
--------------------------------------------------


separator优先原则：当设置了 separator （如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断。
这种设计是为了：
1. 优先保持语义完整性（不切断句子）
2. 避免产生无意义的碎片（如半个单词/不完整句子）
3. 如果 chunk_size 比片段小，无法拆分片段，导致 overlap失效。
4. chunk_overlap仅在合并后的片段之间生效（如果 chunk_size 足够大）。如果没有合并的片段，
则 overlap失效。

###  RecursiveCharacterTextSplitter：最常用
递归自负文本切割器，遇到特定字符时进行分割。默认情况下，它尝试进行切割的字符包括 ["\n\n", "\n", " ", ""]

很典型的RAG切分思路：

优先按更`自然的文本边界`切分，若切分后的片段仍过大，再逐级退化到更细粒度的分隔符，以此类推。最后再按 chunk_size 与 chunk_overlap 组织为最终 chunk。

还可以自定义的方式添加，。等分割字符。

特点：

- 保留上下文：优先在自然语言边界（如段落、句子结尾）处分割， 减少信息碎片化 。

- 智能分段：通过递归尝试多种分隔符，将文本分割为大小 接近chunk_size 的片段。
  
- 灵活适配：适用于多种文本类型（代码、Markdown、普通文本等），是LangChain中 最通用 的
文本拆分器。

可以指定的参数包括：

- chunk_size ：同TextSplitter（父类） 。
  
- chunk_overlap ：同TextSplitter（父类） 。
  
- length_function ：同TextSplitter（父类） 。
  
- add_start_index ：同TextSplitter（父类） 。

In [10]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)
# 3.定义拆分的内容
text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"
# 4.拆分器分割
paragraphs = text_splitter.split_text(text)
for i,chunk in enumerate(paragraphs):
    print(f"块{i + 1},长度：{len(chunk)}")
    print(chunk)
    print('-' * 50)

块1,长度：10
LangChain框
--------------------------------------------------
块2,长度：3
架特性
--------------------------------------------------
块3,长度：9
多模型集成(GPT
--------------------------------------------------
块4,长度：8
/Claude)
--------------------------------------------------
块5,长度：6
记忆管理功能
--------------------------------------------------
块6,长度：9
链式调用设计。文档
--------------------------------------------------
块7,长度：10
分析场景示例：需要处
--------------------------------------------------
块8,长度：10
理PDF/Word等
--------------------------------------------------
块9,长度：3
格式。
--------------------------------------------------


In [ ]:
# 使用create_documents()方法演示，传入字符串列表，返回Document对象列表
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义分割的内容
# text = "LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"
texts = [
    "LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"
]

# 4.分割器分割
# create_documents()：形参是字符串列表，返回值是Document的列表
paragraphs = text_splitter.create_documents(texts)
for para in paragraphs:
    print(para)
    print("-------")

## 切割过程
首先按照\n\n进行分割
如果分割后长度大于对应的chunk_size，则继续按照\n进行分割
如果还是大于，则尝试空格分割
如果空格分割后还是大于，则尝试递归分割

In [11]:
# 使用create_documents()方法演示，将本地文件内容加载成字符串，进行拆分
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.打开.txt文件
with open("../assets/load/09-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  # 返回的是字符串

# 3.定义RecursiveCharacterTextSplitter（递归字符分割器）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    # chunk_overlap=0,
    length_function=len,
)

# 4.分割文本
texts = text_splitter.create_documents([state_of_the_union])

# 5.打印分割文本
for text in texts:
    print(f"🔥{text.page_content}")

🔥人工智能（AI）是什么？
🔥人工智能（Artificial
🔥Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
🔥让机器具备感知环境、处理信息并自主行动的能力。
🔥1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
🔥深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
🔥2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
🔥金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
🔥伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。
🔥未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。


In [ ]:
# 使用split_documents()方法演示，利用PDFLoader加载文档，对文档的内容用递归切割器切割
# 1.导入相关依赖
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义PyPDFLoader加载器
loader = PyPDFLoader("../assets/load/04-load.pdf")

# 3.加载和切割文档对象
docs = loader.load()  # 返回Document对象构成的list
# print(f"第0页：\n{docs[0]}")

# 4.定义切割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    # chunk_size=120,
    chunk_overlap=0,
    # chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

# 5.对pdf内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)
for para in paragraphs:
    print(para)
    print("-------")

#### RecursiveCharacterTextSplitter 底层处理逻辑
##### 先拆分
1. 底层的 self._split_text() 按照分隔符列表的顺序应用当前递归层可用的第一个分隔符，将文档切
成若干块。

2. 如果切分后的块大小 >chunk_size 则调用 self._split_text() 用下一个分隔符递归处理大块
   
3. 直至所有块大小都不超过 chunk_size ，停止递归

##### 后合并
合并过程不是一次完成的，为便于理解抽象为一次合并，最终得到的完整块列表为 final_chunks

1. 遍历切分后的 chunk 列表。对每个当前 chunk，先判断若将其加入候选窗口后，是否会使候选窗口长度（包含必要的合并分隔符（默认为""））超过 chunk_size 。若超过，则先将历史块合并为整体，添加到 final_chunks 。
   
2. 然后从候选列表左侧逐个弹出chunk，直至：
   1. 剩余chunk拼接后的累计长度不大于 chunk_overlap
   2. 并且剩余部分+下一个chunk的长度+合并分隔符长度之和不超过 chunk_size
   3. 这样可以让拆分过细的小块同时出现在前后两个相邻的块中
   
3. 处理完成后，就得到了满足 chunk_size 约束，并且按照 chunk_overlap 保留重叠区域的chunk 列表

# TokenTextSplitter/CharacterTextSplitter：Split by 

按Token的数量分割 

TokenTextSplitter 使用说明：

- 核心依据：Token数量 + 自然边界。（TokenTextSplitter 严格按照 token 数量进行分割，但同时会优先在自然边界（如句尾）处切断，以尽量保证语义的完整性。）
  
- 优点：与LLM的Token计数逻辑一致，能尽量保持语义完整
  
- 缺点：对非英语或特定领域文本，Token化效果可能不佳

- 典型场景 ：需要精确控制Token数输入LLM的场景

TokenTextSplitter 底层会用到 token 编码器，后者的主要功能是将输入的文本切分为token序列，
并将token序列映射为ID序列，本质上是一个 tokenizer

In [12]:
# 1.导入相关依赖
from langchain_text_splitters import TokenTextSplitter

# 2.初始化 TokenTextSplitter
text_splitter = TokenTextSplitter(
    chunk_size=33,  # 最大 token 数为 33
    chunk_overlap=0,  # 重叠 token 数为 0
    # model_name="gpt-4",  # 选择 GPT-4 模型的编码器
    encoding_name="cl100k_base",  # 使用 OpenAI 的编码器,将文本转换为 token 序列
)

# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏。"

# 4.开始切割
texts = text_splitter.split_text(text)

# 打印分割结果
print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

原始文本被分割成了 3 个块:
块 1: 长度：29 内容：人工智能是一个强大的开发框架。它支持多种语言模型和工具链。
--------------------------------------------------
块 2: 长度：32 内容：人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50
--------------------------------------------------
块 3: 长度：19 内容：年代诞生以来，人工智能经历了多次起伏。
--------------------------------------------------


###### 参数说明

可选编码器位于 openai_public.py 文件的全局变量中，如下所示：

ENCODING_CONSTRUCTORS = {

"gpt2": gpt2,

"r50k_base": r50k_base,

"p50k_base": p50k_base,

"p50k_edit": p50k_edit,

"cl100k_base": cl100k_base,

"o200k_base": o200k_base,

"o200k_harmony": o200k_harmony,

}

####  SemanticChunker：语义分块
SemanticChunking（语义分块）是 LangChain 中一种更高级的文本分割方法，它超越了传统的基于字符或固定大小的分块方式，而是根据 文本的语义结构 进行智能分块，使每个分块保持 语义完整性 ，从而提高检索增强生成(RAG)等应用的效果。

通过将文本转化为 向量（Embedding） ，去计算前后句子的语义差异，当发现前后两句话的语义变化很大（超过设定的阈值）时，就在这里一刀切断。

In [19]:
# pip install langchain_experimental
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings
from langchain_experimental.text_splitter import SemanticChunker

load_dotenv(override=True)

# 1.加载文本
with open("../assets/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  # 返回字符串

# pip install langchain_experimental
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings
from langchain_experimental.text_splitter import SemanticChunker

load_dotenv(override=True)

# 1.加载文本
with open("../assets/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  # 返回字符串

# 2.获取嵌入模型（用于计算句子间语义距离）
# 注意：base_url 应为 OpenAI 兼容根地址（如 https://api.302.ai/v1），不要带 /embeddings
embedding_base = os.getenv("EMBEDDING_API_BASE") or os.getenv("OPEN_302_API_BASE")
if embedding_base and embedding_base.rstrip("/").endswith("/embeddings"):
    embedding_base = embedding_base.rstrip("/").removesuffix("/embeddings")

embedding_model = init_embeddings(
    model="openai:text-embedding-3-large",
    api_key=os.getenv("EMBEDDING_API_KEY") or os.getenv("OPEN_302_API_KEY"),
    base_url=embedding_base,
)

# 3.获取切割器
# breakpoint_threshold_type 可选:
#   "percentile" | "standard_deviation" | "interquartile" | "gradient"
text_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",  # 按语义距离百分位切分
    breakpoint_threshold_amount=65.0,  # 阈值越低，切分越细（块越多）
    # 中文文本句号后通常无空格，用 \s* 兼容有无空格两种情况
    sentence_split_regex=r"(?<=[。？！.?!])\s*",
)

# 4.切分文档
docs = text_splitter.create_documents([state_of_the_union])
print(f"共切分为 {len(docs)} 个语义块\n")
for i, doc in enumerate(docs, start=1):
    print(f"🔍 块 {i} | 长度={len(doc.page_content)}")
    print(doc.page_content)
    print("-" * 50)

# 3.获取切割器
# breakpoint_threshold_type 可选:
#   "percentile" | "standard_deviation" | "interquartile" | "gradient"
text_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",  # 按语义距离百分位切分
    breakpoint_threshold_amount=65.0,  # 阈值越低，切分越细（块越多）
    # 中文文本句号后通常无空格，用 \s* 兼容有无空格两种情况
    sentence_split_regex=r"(?<=[。？！.?!])\s*",
)

# 4.切分文档
docs = text_splitter.create_documents([state_of_the_union])
print(f"共切分为 {len(docs)} 个语义块\n")
for i, doc in enumerate(docs, start=1):
    print(f"🔍 块 {i} | 长度={len(doc.page_content)}")
    print(doc.page_content)
    print("-" * 50)

共切分为 26 个语义块

🔍 块 1 | 长度=245
人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence，AI）作为计算机科学的一个重要分支，近年来取得了突飞猛进的发展。 本文综述了人工智能的发展历程、核心技术、应用领域以及未来发展趋势。 通过对人工智能的定义、历史背景、主要技术（如机器学习、深度学习、自然语言处理等）的详细介绍，探讨了人工智能在医疗、金融、教育、交通等领域的应用，并分析了人工智能发展过程中面临的挑战与机遇。 最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方向。
--------------------------------------------------
🔍 块 2 | 长度=2
1.
--------------------------------------------------
🔍 块 3 | 长度=164
引言
人工智能是指通过计算机程序模拟人类智能的一门科学。 自20世纪50年代诞生以来，人工智能经历了多次起伏，近年来随着计算能力的提升和大数据的普及，人工智能技术取得了显著的进展。 人工智能的应用已经渗透到日常生活的方方面面，从智能手机的语音助手到自动驾驶汽车，从医疗诊断到金融分析，人工智能正在改变着人类社会的运行方式。 2.
--------------------------------------------------
🔍 块 4 | 长度=121
人工智能的发展历程
2. 1 早期发展
人工智能的概念最早可以追溯到20世纪50年代。 1956年，达特茅斯会议（Dartmouth Conference）被认为是人工智能研究的正式开端。 在随后的几十年里，人工智能研究经历了多次高潮与低谷。
--------------------------------------------------
🔍 块 5 | 长度=43
早期的研究主要集中在符号逻辑和专家系统上，但由于计算能力的限制和算法的不足，进展缓慢。
--------------------------------------------------
🔍 块 6 | 长度=2
2.
--------------------------------------------------
🔍 块 7 | 长度=1